# EDA


In [ ]:
import warnings
from pathlib import Path
import random

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")


## 1. Подготовка


Базовое окружение для анализа собрано: подключены библиотеки для таблиц, массивов, путей и расчета RMSE. Предупреждения отключены, поэтому вывод ноутбука остается компактным.


In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path(".")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

TARGETS_RAW = ["IC50, mM", "CC50, mM", "SI"]
TARGETS = ["IC50", "CC50", "SI"]
ID_COL = "index"


## 2. Конфигурация


Пути к данным, папка для результатов, список таргетов и seed заданы в одном месте. Это снижает риск ошибок в названиях колонок и делает запуск ноутбука воспроизводимым.


In [ ]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def competition_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    scores = {
        "IC50": rmse(y_true[:, 0], y_pred[:, 0]),
        "CC50": rmse(y_true[:, 1], y_pred[:, 1]),
        "SI": rmse(y_true[:, 2], y_pred[:, 2]),
    }

    return float(np.mean(list(scores.values()))), scores


def make_row_hash(df):
    tmp = df.copy().fillna(-999999999)
    return pd.util.hash_pandas_object(tmp, index=False)


## 3. Вспомогательные функции


Метрика качества считается отдельно по IC50, CC50 и SI, а затем усредняется. Хеширование строк подготовлено для диагностики полных совпадений между train и test.


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = train.rename(columns={
    "IC50, mM": "IC50",
    "CC50, mM": "CC50",
})

feature_cols = [c for c in test.columns if c != ID_COL]

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGETS].astype(float).copy()
y_values = y.values

print("=" * 80)
print("DATA SHAPES")
print("=" * 80)
print("train:", train.shape)
print("test:", test.shape)
print("X:", X.shape)
print("X_test:", X_test.shape)
print("y:", y.shape)


## 4. Загрузка данных и формирование матриц


Данные успешно разделены на признаки train, признаки test и три целевые переменные. Размеры таблиц дают быстрый контроль, что чтение файлов и выбор колонок прошли корректно.


In [ ]:
print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing_info = pd.DataFrame({
    "dataset": ["X_train", "X_test", "y"],
    "total_missing": [
        int(X.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
        int(y.isna().sum().sum()),
    ],
    "columns_with_missing": [
        int((X.isna().sum() > 0).sum()),
        int((X_test.isna().sum() > 0).sum()),
        int((y.isna().sum() > 0).sum()),
    ],
})

display(missing_info)


## 5. Пропуски


Информация о пропусках собрана отдельно для обучающих признаков, тестовых признаков и таргетов. Этот результат показывает, нужна ли дополнительная обработка пустых значений перед моделированием.


In [ ]:
print("\n" + "=" * 80)
print("TARGET STATISTICS")
print("=" * 80)

target_stats = y.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
display(target_stats)


## 6. Статистика таргетов


Распределения целевых переменных выглядят сложными для моделирования. Особенно выделяется SI: максимум 15620.6 при медиане 4.0, что говорит о сильной скошенности и наличии выбросов.


In [ ]:
print("\n" + "=" * 80)
print("SI FORMULA CHECK")
print("=" * 80)

si_formula = train["CC50"] / train["IC50"]
si_diff = (train["SI"] - si_formula).abs()

print("SI = CC50 / IC50")
print("max diff:", si_diff.max())
print("mean diff:", si_diff.mean())
print("median diff:", si_diff.median())

si_formula_check = pd.DataFrame({
    "metric": ["max_abs_diff", "mean_abs_diff", "median_abs_diff"],
    "value": [si_diff.max(), si_diff.mean(), si_diff.median()],
})

display(si_formula_check)


## 7. Формула SI


Доменное соотношение подтвердилось: SI практически точно равен CC50 / IC50. Максимальная ошибка составляет около 2e-11, поэтому эту связь можно учитывать в финальном решении.


In [ ]:
print("\n" + "=" * 80)
print("CONSTANT FEATURES")
print("=" * 80)

constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X_noconst = X.drop(columns=constant_cols)
X_test_noconst = X_test.drop(columns=constant_cols)

print("constant features:", len(constant_cols))
print("X_noconst:", X_noconst.shape)
print("X_test_noconst:", X_test_noconst.shape)

if len(constant_cols) > 0:
    display(pd.DataFrame({"constant_feature": constant_cols}))


## 8. Константные признаки


В train найдено 18 константных признаков. Такие колонки не добавляют информации для моделей, поэтому их можно удалить перед частью экспериментов.


In [ ]:
print("\n" + "=" * 80)
print("DUPLICATE / EXACT-MATCH DIAGNOSTICS")
print("=" * 80)

train_hash = make_row_hash(X_noconst)
test_hash = make_row_hash(X_test_noconst)

train_group_sizes = train_hash.value_counts()
test_has_exact = test_hash.isin(set(train_hash))

duplicate_stats = {
    "unique_train_feature_groups": int(train_hash.nunique()),
    "train_duplicate_groups": int((train_group_sizes > 1).sum()),
    "train_rows_in_duplicate_groups": int(train_hash.isin(train_group_sizes[train_group_sizes > 1].index).sum()),
    "test_rows_with_exact_train_match": int(test_has_exact.sum()),
    "test_exact_match_share": float(test_has_exact.mean()),
}

for k, v in duplicate_stats.items():
    print(f"{k}: {v}")

duplicate_stats_df = pd.DataFrame(
    list(duplicate_stats.items()),
    columns=["metric", "value"]
)

display(duplicate_stats_df)


## 9. Дубликаты и exact-match


В test есть 68 объектов, которые полностью совпадают с объектами из train по молекулярным дескрипторам. Из-за таких повторов локальная CV-валидация и результат на Kaggle могут заметно расходиться.


In [ ]:
print("\n" + "=" * 80)
print("MEDIAN BASELINE")
print("=" * 80)

median_pred = np.tile(y.median().values, (len(y), 1))
median_score, median_per_target = competition_score(y_values, median_pred)

print("median baseline score:", median_score)
print(median_per_target)

baseline_df = pd.DataFrame({
    "target": list(median_per_target.keys()),
    "rmse": list(median_per_target.values()),
})

display(baseline_df)


## 10. Median baseline


Простая стратегия с предсказанием медианы дает RMSE около 622.72. Это нижняя планка качества, которую должны существенно улучшить дальнейшие модели и ансамбли.


In [ ]:
analytics_summary = {
    "train_shape": train.shape,
    "test_shape": test.shape,
    "features_count": len(feature_cols),
    "target_columns": TARGETS,
    "missing_train_total": int(X.isna().sum().sum()),
    "missing_test_total": int(X_test.isna().sum().sum()),
    "constant_features_count": len(constant_cols),
    "si_formula_max_diff": float(si_diff.max()),
    "si_formula_mean_diff": float(si_diff.mean()),
    "duplicate_stats": duplicate_stats,
    "median_baseline_score": float(median_score),
    "median_baseline_per_target": {
        k: float(v) for k, v in median_per_target.items()
    },
}

pd.Series(analytics_summary).to_json(
    OUT_DIR / "stage1_analytics_summary.json",
    force_ascii=False,
    indent=2,
)

target_stats.to_csv(OUT_DIR / "stage1_target_stats.csv")
missing_info.to_csv(OUT_DIR / "stage1_missing_info.csv", index=False)
duplicate_stats_df.to_csv(OUT_DIR / "stage1_duplicate_stats.csv", index=False)
baseline_df.to_csv(OUT_DIR / "stage1_median_baseline.csv", index=False)

print("\nSaved analytics files to outputs/")
print("Stage 1 finished OK")


## 11. Сохранение диагностик


Основные артефакты EDA сохранены в outputs: общая сводка, статистика таргетов, информация о пропусках, дубликатах и медианном baseline. Эти файлы можно использовать на следующих этапах.


## Итоговый вывод


Главные наблюдения после EDA:

- таргеты заметно выбросные: у SI максимум 15620.6, медиана 4.0;
- SI фактически считается как CC50 / IC50, максимальная ошибка около 2e-11;
- в train есть 18 константных признаков;
- 68 test-объектов имеют точное совпадение в train;
- медианный baseline дает RMSE около 622.72.

Валидацию стоит делать аккуратно, а финальное решение лучше строить с учетом формулы для SI и возможных exact-match объектов.
